In [ ]:
from __future__ import annotations

import gc
import sys
from pathlib import Path

import numpy as np
import pandas as pd

try:
    import torch
except Exception:  # pragma: no cover
    torch = None


OUTER_FOLDS = (1, 2, 3, 4, 5)
WASTE_TYPES = ("AW", "CDW", "IW", "MSW")
MODEL_FAMILY = "tabpfn"
POOLED_MODEL_ID = "tabpfn_v26_core14"
PER_WASTE_MODEL_ID = "tabpfn_v26_per_waste_core10"
RUN_ROLE = "default"
TARGET_COL = "Target_Log"
TARGET_RAW_COL = "Waste_Generation_t"
TABPFN_N_ESTIMATORS = 8
TABPFN_SUBSAMPLE_SAMPLES = 2000
SEED = 42
BATCH_SIZE = 512
PREDICT_PROGRESS_EVERY = 2
EXPECTED_DRIVER_FEATURE_COUNT = 10
EXPECTED_ROUTE_FEATURES = ["WasteRoute_AW", "WasteRoute_CDW", "WasteRoute_IW", "WasteRoute_MSW"]


build_prediction_frame = None
compute_regression_metrics = None
save_model_artifact = None
TabPFNRegressor = None
ModelVersion = None


In [ ]:
def resolve_code_dir() -> Path:
    if "__file__" in globals():
        return Path(__file__).resolve().parent
    code_dir = Path.cwd().resolve()
    if code_dir.name != "1_Code":
        raise RuntimeError("current working directory must be Step6_TabPFNExplainability/1_Code")
    return code_dir


CODE_DIR = resolve_code_dir()
STEP_ROOT = CODE_DIR.parent
RELEASE_ROOT = STEP_ROOT.parent
STEP4_ROOT = RELEASE_ROOT / "Step4_ModelInputs" / "2_Results"
STEP5_ROOT = RELEASE_ROOT / "Step5_ModelComparison"
RESULT_DIR = STEP_ROOT / "2_Results"
ARTIFACT_DIR = RESULT_DIR / "2_per_waste_core10_artifacts"
RESULT_DIR.mkdir(parents=True, exist_ok=True)
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

POOLED_PREDICTION_PATH = RESULT_DIR / "1_core_feature_ablation_predictions_outer_test.csv"
PER_WASTE_PREDICTION_PATH = RESULT_DIR / "2_per_waste_core10_predictions_outer_test.csv"
METRICS_BY_FOLD_PATH = RESULT_DIR / "2_pooled_vs_per_waste_metrics_by_waste_fold.csv"
METRICS_SUMMARY_PATH = RESULT_DIR / "2_pooled_vs_per_waste_metrics_by_waste_summary.csv"
DIAGNOSTICS_PATH = RESULT_DIR / "2_waste_heterogeneity_diagnostics.csv"
FEATURE_CONTRACT_PATH = RESULT_DIR / "2_per_waste_core10_feature_contract.csv"
ARTIFACT_MANIFEST_PATH = RESULT_DIR / "2_per_waste_core10_model_artifact_manifest.csv"

if not STEP4_ROOT.is_dir():
    raise FileNotFoundError(f"cannot resolve Step4 results from {STEP4_ROOT}")
if not STEP5_ROOT.is_dir():
    raise FileNotFoundError(f"cannot resolve Step5 root from {STEP5_ROOT}")

print({"step": "paths_ready", "result_dir": str(RESULT_DIR), "artifact_dir": str(ARTIFACT_DIR)})


In [ ]:
def initialize_runtime() -> None:
    global build_prediction_frame
    global compute_regression_metrics
    global save_model_artifact
    global TabPFNRegressor
    global ModelVersion

    step5_code_dir = STEP5_ROOT / "1_Code"
    if str(step5_code_dir) not in sys.path:
        sys.path.insert(0, str(step5_code_dir))

    from _step5_shared import (
        TARGET_COL as shared_target_col,
        TARGET_RAW_COL as shared_target_raw_col,
        build_prediction_frame as shared_build_prediction_frame,
        compute_regression_metrics as shared_compute_regression_metrics,
        save_model_artifact as shared_save_model_artifact,
    )

    if shared_target_col != TARGET_COL:
        raise RuntimeError(f"unexpected target column from Step5 shared module: {shared_target_col}")
    if shared_target_raw_col != TARGET_RAW_COL:
        raise RuntimeError(f"unexpected raw target column from Step5 shared module: {shared_target_raw_col}")

    from tabpfn import TabPFNRegressor as runtime_tabpfn_regressor
    from tabpfn.constants import ModelVersion as runtime_model_version

    supported_versions = [name for name in dir(runtime_model_version) if name.startswith("V")]
    if "V2_6" not in supported_versions:
        raise RuntimeError("tabpfn runtime must expose ModelVersion.V2_6; version fallback is not allowed")

    if torch is not None:
        torch.set_num_threads(max(1, min(8, torch.get_num_threads())))

    build_prediction_frame = shared_build_prediction_frame
    compute_regression_metrics = shared_compute_regression_metrics
    save_model_artifact = shared_save_model_artifact
    TabPFNRegressor = runtime_tabpfn_regressor
    ModelVersion = runtime_model_version
    print({"step": "runtime_ready", "tabpfn_versions": supported_versions})


initialize_runtime()


In [ ]:
def require_file(path: Path) -> Path:
    if not path.is_file():
        raise FileNotFoundError(str(path))
    return path


def read_feature_list(path: Path) -> list[str]:
    frame = pd.read_csv(require_file(path))
    if "feature" not in frame.columns:
        raise KeyError(f"feature column missing from {path}")
    features = frame["feature"].astype(str).tolist()
    if len(features) != len(set(features)):
        raise RuntimeError(f"feature contract contains duplicates: {path}")
    return features


DRIVER_FEATURES = read_feature_list(STEP4_ROOT / "0_feature_contract_10feature.csv")
if len(DRIVER_FEATURES) != EXPECTED_DRIVER_FEATURE_COUNT:
    raise RuntimeError(f"Step4 locked ten-feature contract must contain 10 driver features, got {len(DRIVER_FEATURES)}")

fold_core_feature_lists = []
for outer_fold in OUTER_FOLDS:
    fold_contract_path = STEP4_ROOT / f"fold_{int(outer_fold)}" / "ten_feature_unified" / "processed" / "0_feature_columns.csv"
    fold_core_feature_lists.append(read_feature_list(fold_contract_path))

CORE14_FEATURES = fold_core_feature_lists[0]
contract_drift = {
    int(fold): features for fold, features in zip(OUTER_FOLDS, fold_core_feature_lists) if features != CORE14_FEATURES
}
if contract_drift:
    raise RuntimeError(f"Step4 ten-feature unified feature contract differs across folds: {contract_drift}")
if CORE14_FEATURES[:EXPECTED_DRIVER_FEATURE_COUNT] != DRIVER_FEATURES:
    raise RuntimeError("Step4 ten-feature unified driver columns do not match the locked ten-feature contract")
if CORE14_FEATURES[EXPECTED_DRIVER_FEATURE_COUNT:] != EXPECTED_ROUTE_FEATURES:
    raise RuntimeError("Step4 ten-feature unified route columns do not match the expected route feature contract")

pd.DataFrame({"feature": DRIVER_FEATURES}).to_csv(FEATURE_CONTRACT_PATH, index=False)
print({"step": "feature_contract_ready", "driver_count": len(DRIVER_FEATURES), "core14_count": len(CORE14_FEATURES)})


In [ ]:
def load_fold_waste_view(outer_fold: int, waste_type: str) -> tuple[pd.DataFrame, pd.DataFrame, list[str]]:
    if waste_type not in WASTE_TYPES:
        raise ValueError(f"unexpected waste_type: {waste_type}")
    processed_dir = STEP4_ROOT / f"fold_{int(outer_fold)}" / "ten_feature_unified" / "processed"
    train_df = pd.read_csv(require_file(processed_dir / "0_train_processed.csv"))
    test_df = pd.read_csv(require_file(processed_dir / "0_test_processed.csv"))
    feature_cols = read_feature_list(processed_dir / "0_feature_columns.csv")
    if feature_cols != CORE14_FEATURES:
        raise RuntimeError(f"fold {outer_fold} core14 contract drift detected")

    required_columns = ["row_uid", "Country Code", "Year", "WasteFlag", "outer_fold", TARGET_RAW_COL, TARGET_COL, *DRIVER_FEATURES]
    missing_train = [column for column in required_columns if column not in train_df.columns]
    missing_test = [column for column in required_columns if column not in test_df.columns]
    if missing_train or missing_test:
        raise KeyError(f"fold {outer_fold} input missing columns: train={missing_train}, test={missing_test}")

    train_part = train_df[train_df["WasteFlag"].astype(str).eq(waste_type)].copy().reset_index(drop=True)
    test_part = test_df[test_df["WasteFlag"].astype(str).eq(waste_type)].copy().reset_index(drop=True)
    if train_part.empty or test_part.empty:
        raise RuntimeError(f"fold {outer_fold} {waste_type} train/test subset must not be empty")
    test_outer_folds = sorted(test_part["outer_fold"].dropna().astype(int).unique().tolist())
    if test_outer_folds != [int(outer_fold)]:
        raise RuntimeError(f"fold {outer_fold} {waste_type} test subset has unexpected outer_fold values: {test_outer_folds}")
    if train_part["row_uid"].astype(str).duplicated().any() or test_part["row_uid"].astype(str).duplicated().any():
        raise RuntimeError(f"fold {outer_fold} {waste_type} subset contains duplicate row_uid values")
    overlap = set(train_part["row_uid"].astype(str)).intersection(set(test_part["row_uid"].astype(str)))
    if overlap:
        raise RuntimeError(f"fold {outer_fold} {waste_type} train/test row_uid overlap detected: overlap_count={len(overlap)}")
    return train_part, test_part, DRIVER_FEATURES


def prepare_tabpfn_feature_frames(
    train_df: pd.DataFrame,
    test_df: pd.DataFrame,
    feature_cols: list[str],
) -> tuple[pd.DataFrame, pd.DataFrame, pd.Series]:
    x_train = train_df.loc[:, list(feature_cols)].apply(pd.to_numeric, errors="coerce")
    x_test = test_df.loc[:, list(feature_cols)].apply(pd.to_numeric, errors="coerce")
    medians = x_train.median(axis=0).fillna(0.0)
    x_train = x_train.fillna(medians)
    x_test = x_test.fillna(medians)
    y_train = pd.Series(pd.to_numeric(train_df[TARGET_COL], errors="raise"), index=train_df.index).astype(float)
    if not np.isfinite(x_train.to_numpy(dtype=float)).all():
        raise ValueError("training feature matrix contains non-finite values")
    if not np.isfinite(x_test.to_numpy(dtype=float)).all():
        raise ValueError("test feature matrix contains non-finite values")
    if not np.isfinite(y_train.to_numpy(dtype=float)).all():
        raise ValueError("training target contains non-finite values")
    return x_train, x_test, y_train


print({"step": "fold_waste_loader_ready"})


In [ ]:
def build_tabpfn_regressor(random_seed: int):
    device = "cuda" if torch is not None and torch.cuda.is_available() else "cpu"
    return TabPFNRegressor.create_default_for_version(
        ModelVersion.V2_6,
        n_estimators=TABPFN_N_ESTIMATORS,
        device=device,
        fit_mode="low_memory",
        memory_saving_mode="auto",
        ignore_pretraining_limits=True,
        inference_config={"SUBSAMPLE_SAMPLES": TABPFN_SUBSAMPLE_SAMPLES},
        random_state=int(random_seed),
    )


def predict_in_batches(model, features: pd.DataFrame, batch_size: int = BATCH_SIZE, progress_every: int = PREDICT_PROGRESS_EVERY) -> np.ndarray:
    n_rows = len(features)
    if n_rows == 0:
        return np.asarray([], dtype=float)
    if batch_size is None or int(batch_size) <= 0 or int(batch_size) >= n_rows:
        return np.asarray(model.predict(features), dtype=float).reshape(-1)
    predictions: list[np.ndarray] = []
    n_batches = (n_rows - 1) // int(batch_size) + 1
    for batch_idx in range(n_batches):
        start = batch_idx * int(batch_size)
        end = min((batch_idx + 1) * int(batch_size), n_rows)
        batch_features = features.iloc[start:end]
        batch_pred = np.asarray(model.predict(batch_features), dtype=float).reshape(-1)
        predictions.append(batch_pred)
        if progress_every and (((batch_idx + 1) % int(progress_every) == 0) or (batch_idx + 1 == n_batches)):
            print(f"Prediction batch {batch_idx + 1}/{n_batches} ({end}/{n_rows} rows)", flush=True)
    return np.concatenate(predictions, axis=0)


def clear_runtime_memory() -> None:
    gc.collect()
    if torch is not None and torch.cuda.is_available():
        torch.cuda.empty_cache()


print({"step": "model_helpers_ready", "n_estimators": TABPFN_N_ESTIMATORS, "subsample_samples": TABPFN_SUBSAMPLE_SAMPLES})


In [ ]:
def load_pooled_core14_predictions() -> pd.DataFrame:
    pooled = pd.read_csv(require_file(POOLED_PREDICTION_PATH))
    required_columns = [
        "outer_fold",
        "model_family",
        "model_id",
        "run_role",
        "row_uid",
        "Country Code",
        "Year",
        "WasteFlag",
        "Actual_Log",
        "Actual_Raw",
        "Prediction_Log",
        "Prediction_Raw",
    ]
    missing = [column for column in required_columns if column not in pooled.columns]
    if missing:
        raise KeyError(f"pooled core14 prediction file missing columns: {missing}")
    if sorted(pooled["outer_fold"].dropna().astype(int).unique().tolist()) != list(OUTER_FOLDS):
        raise RuntimeError("pooled core14 predictions must cover folds 1..5")
    if sorted(pooled["WasteFlag"].dropna().astype(str).unique().tolist()) != sorted(WASTE_TYPES):
        raise RuntimeError("pooled core14 predictions must cover the four expected waste types")
    if not pooled["model_id"].astype(str).eq(POOLED_MODEL_ID).all():
        raise RuntimeError(f"pooled prediction model_id must be {POOLED_MODEL_ID}")
    if pooled["row_uid"].astype(str).duplicated().any():
        raise RuntimeError("pooled core14 predictions contain duplicate row_uid values")
    return pooled


pooled_core14_predictions = load_pooled_core14_predictions()
print({"step": "pooled_predictions_ready", "rows": int(len(pooled_core14_predictions))})


In [ ]:
def run_one_waste_bridge(waste_type: str) -> dict[str, object]:
    prediction_frames = []
    metric_rows = []
    artifact_rows = []
    for outer_fold in OUTER_FOLDS:
        train_df, test_df, feature_cols = load_fold_waste_view(int(outer_fold), waste_type)
        x_train, x_test, y_train = prepare_tabpfn_feature_frames(train_df, test_df, feature_cols)
        print(
            {
                "step": "waste_fold_start",
                "waste_type": waste_type,
                "outer_fold": int(outer_fold),
                "train_rows": int(len(x_train)),
                "test_rows": int(len(x_test)),
                "features": int(len(feature_cols)),
            },
            flush=True,
        )
        model = build_tabpfn_regressor(random_seed=SEED)
        model.fit(x_train, y_train.to_numpy(dtype=float))
        prediction_log = predict_in_batches(model, x_test, batch_size=BATCH_SIZE, progress_every=PREDICT_PROGRESS_EVERY)
        prediction_frame = build_prediction_frame(
            test_df=test_df,
            prediction_log=prediction_log,
            outer_fold=int(outer_fold),
            model_family=MODEL_FAMILY,
            model_id=PER_WASTE_MODEL_ID,
            run_role=RUN_ROLE,
        )
        metric_row = {
            "model_scope": "per_waste_core10",
            "WasteFlag": waste_type,
            "outer_fold": int(outer_fold),
            "model_family": MODEL_FAMILY,
            "model_id": PER_WASTE_MODEL_ID,
            "run_role": RUN_ROLE,
            **compute_regression_metrics(prediction_frame),
            "rows": int(len(prediction_frame)),
        }
        artifact_path = ARTIFACT_DIR / waste_type / f"fold_{int(outer_fold)}" / f"{PER_WASTE_MODEL_ID}_{RUN_ROLE}.joblib"
        save_model_artifact(model, artifact_path)
        if not artifact_path.is_file():
            raise RuntimeError(f"artifact path does not exist after save: {artifact_path}")
        artifact_rows.append(
            {
                "WasteFlag": waste_type,
                "outer_fold": int(outer_fold),
                "model_family": MODEL_FAMILY,
                "model_id": PER_WASTE_MODEL_ID,
                "run_role": RUN_ROLE,
                "feature_count": len(feature_cols),
                "seed": SEED,
                "tabpfn_n_estimators": TABPFN_N_ESTIMATORS,
                "tabpfn_subsample_samples": TABPFN_SUBSAMPLE_SAMPLES,
                "batch_size": BATCH_SIZE,
                "artifact_path": str(artifact_path),
            }
        )
        prediction_frames.append(prediction_frame)
        metric_rows.append(metric_row)
        del model
        clear_runtime_memory()
        print({"step": "waste_fold_completed", "waste_type": waste_type, "outer_fold": int(outer_fold), **metric_row}, flush=True)
    return {
        "prediction_frame": pd.concat(prediction_frames, ignore_index=True),
        "metric_rows": metric_rows,
        "artifact_rows": artifact_rows,
    }


per_waste_results_by_waste: dict[str, dict[str, object]] = {}
print({"step": "per_waste_runner_ready"})


In [ ]:
per_waste_results_by_waste["AW"] = run_one_waste_bridge("AW")


In [ ]:
per_waste_results_by_waste["CDW"] = run_one_waste_bridge("CDW")


In [ ]:
per_waste_results_by_waste["IW"] = run_one_waste_bridge("IW")


In [ ]:
per_waste_results_by_waste["MSW"] = run_one_waste_bridge("MSW")


In [ ]:
def write_per_waste_outputs() -> tuple[pd.DataFrame, pd.DataFrame]:
    completed_wastes = sorted(per_waste_results_by_waste)
    if completed_wastes != sorted(WASTE_TYPES):
        raise RuntimeError(f"cannot finalize before all waste types are completed: completed={completed_wastes}")
    prediction_frames = [per_waste_results_by_waste[waste]["prediction_frame"] for waste in WASTE_TYPES]
    metric_rows_nested = [per_waste_results_by_waste[waste]["metric_rows"] for waste in WASTE_TYPES]
    artifact_rows_nested = [per_waste_results_by_waste[waste]["artifact_rows"] for waste in WASTE_TYPES]
    predictions = pd.concat(prediction_frames, ignore_index=True)
    metrics = pd.DataFrame([row for rows in metric_rows_nested for row in rows])
    artifacts = pd.DataFrame([row for rows in artifact_rows_nested for row in rows])
    predictions.to_csv(PER_WASTE_PREDICTION_PATH, index=False)
    artifacts.to_csv(ARTIFACT_MANIFEST_PATH, index=False)
    print({"step": "per_waste_outputs_written", "prediction_rows": int(len(predictions)), "artifact_rows": int(len(artifacts))})
    return predictions, metrics


per_waste_core10_predictions, per_waste_core10_metrics = write_per_waste_outputs()


In [ ]:
def build_pooled_metrics_by_waste_fold(pooled_predictions: pd.DataFrame) -> pd.DataFrame:
    metric_rows = []
    grouped = pooled_predictions.groupby(["WasteFlag", "outer_fold"], sort=True)
    for (waste_type, outer_fold), group in grouped:
        metric_rows.append(
            {
                "model_scope": "pooled_core14",
                "WasteFlag": str(waste_type),
                "outer_fold": int(outer_fold),
                "model_family": MODEL_FAMILY,
                "model_id": POOLED_MODEL_ID,
                "run_role": RUN_ROLE,
                **compute_regression_metrics(group.copy()),
                "rows": int(len(group)),
            }
        )
    pooled_metrics = pd.DataFrame(metric_rows)
    expected_pairs = {(waste, int(fold)) for waste in WASTE_TYPES for fold in OUTER_FOLDS}
    actual_pairs = set(zip(pooled_metrics["WasteFlag"].astype(str), pooled_metrics["outer_fold"].astype(int)))
    if actual_pairs != expected_pairs:
        raise RuntimeError(f"pooled by-waste fold coverage mismatch: actual={sorted(actual_pairs)}, expected={sorted(expected_pairs)}")
    return pooled_metrics


pooled_core14_metrics_by_waste_fold = build_pooled_metrics_by_waste_fold(pooled_core14_predictions)
bridge_metrics_by_fold = pd.concat([pooled_core14_metrics_by_waste_fold, per_waste_core10_metrics], ignore_index=True)
bridge_metrics_by_fold.to_csv(METRICS_BY_FOLD_PATH, index=False)
print({"step": "bridge_metrics_by_fold_written", "rows": int(len(bridge_metrics_by_fold))})


In [ ]:
def build_bridge_summary(metrics_by_fold: pd.DataFrame) -> pd.DataFrame:
    summary = (
        metrics_by_fold.groupby(["WasteFlag", "model_scope", "model_family", "model_id", "run_role"], as_index=False)
        .agg(
            WAPE_micro_mean=("WAPE_micro", "mean"),
            WAPE_micro_std=("WAPE_micro", "std"),
            R2_log_mean=("R2_log", "mean"),
            R2_log_std=("R2_log", "std"),
            WAPE_macro_mean=("WAPE_macro", "mean"),
            WAPE_macro_std=("WAPE_macro", "std"),
            R2_original_mean=("R2_original", "mean"),
            R2_original_std=("R2_original", "std"),
            MAE_original_mean=("MAE_original", "mean"),
            MAE_original_std=("MAE_original", "std"),
            rows_mean=("rows", "mean"),
        )
        .fillna(0.0)
        .sort_values(["WasteFlag", "model_scope"], kind="mergesort")
        .reset_index(drop=True)
    )
    return summary


bridge_metrics_summary = build_bridge_summary(bridge_metrics_by_fold)
bridge_metrics_summary.to_csv(METRICS_SUMMARY_PATH, index=False)
print({"step": "bridge_metrics_summary_written", "rows": int(len(bridge_metrics_summary))})
print(bridge_metrics_summary.to_string(index=False))


In [ ]:
def build_heterogeneity_diagnostics(
    pooled_predictions: pd.DataFrame,
    metrics_summary: pd.DataFrame,
) -> pd.DataFrame:
    target_stats = (
        pooled_predictions.groupby("WasteFlag", as_index=False)
        .agg(
            observed_rows=("Actual_Raw", "size"),
            target_total_raw=("Actual_Raw", "sum"),
            target_mean_raw=("Actual_Raw", "mean"),
            target_median_raw=("Actual_Raw", "median"),
        )
    )
    total_target = float(target_stats["target_total_raw"].sum())
    if total_target <= 0:
        raise RuntimeError("total target mass must be positive for heterogeneity diagnostics")
    target_stats["target_total_share"] = target_stats["target_total_raw"] / total_target

    wide_metrics = metrics_summary.pivot(index="WasteFlag", columns="model_scope", values=["WAPE_micro_mean", "R2_log_mean"])
    wide_metrics.columns = [f"{metric}_{scope}" for metric, scope in wide_metrics.columns]
    wide_metrics = wide_metrics.reset_index()
    diagnostics = target_stats.merge(wide_metrics, on="WasteFlag", how="left", validate="one_to_one")
    required_metric_cols = [
        "WAPE_micro_mean_pooled_core14",
        "WAPE_micro_mean_per_waste_core10",
        "R2_log_mean_pooled_core14",
        "R2_log_mean_per_waste_core10",
    ]
    missing = [column for column in required_metric_cols if column not in diagnostics.columns]
    if missing:
        raise RuntimeError(f"diagnostics missing metric columns: {missing}")
    diagnostics["delta_per_waste_minus_pooled_WAPE_micro_mean"] = (
        diagnostics["WAPE_micro_mean_per_waste_core10"] - diagnostics["WAPE_micro_mean_pooled_core14"]
    )
    diagnostics["delta_per_waste_minus_pooled_R2_log_mean"] = (
        diagnostics["R2_log_mean_per_waste_core10"] - diagnostics["R2_log_mean_pooled_core14"]
    )
    diagnostics = diagnostics.sort_values("WasteFlag", kind="mergesort").reset_index(drop=True)
    return diagnostics


waste_heterogeneity_diagnostics = build_heterogeneity_diagnostics(pooled_core14_predictions, bridge_metrics_summary)
waste_heterogeneity_diagnostics.to_csv(DIAGNOSTICS_PATH, index=False)
print({"step": "heterogeneity_diagnostics_written", "rows": int(len(waste_heterogeneity_diagnostics))})
print(waste_heterogeneity_diagnostics.to_string(index=False))


In [ ]:
def assert_expected_outputs() -> None:
    expected_paths = [
        PER_WASTE_PREDICTION_PATH,
        METRICS_BY_FOLD_PATH,
        METRICS_SUMMARY_PATH,
        DIAGNOSTICS_PATH,
        FEATURE_CONTRACT_PATH,
        ARTIFACT_MANIFEST_PATH,
    ]
    missing = [str(path) for path in expected_paths if not path.is_file()]
    if missing:
        raise RuntimeError(f"missing expected bridge output files: {missing}")
    predictions = pd.read_csv(PER_WASTE_PREDICTION_PATH)
    if predictions.empty:
        raise RuntimeError("per-waste predictions must not be empty")
    if sorted(predictions["outer_fold"].dropna().astype(int).unique().tolist()) != list(OUTER_FOLDS):
        raise RuntimeError("per-waste predictions must cover folds 1..5")
    if sorted(predictions["WasteFlag"].dropna().astype(str).unique().tolist()) != sorted(WASTE_TYPES):
        raise RuntimeError("per-waste predictions must cover the four expected waste types")
    if predictions["row_uid"].astype(str).duplicated().any():
        raise RuntimeError("per-waste predictions contain duplicate row_uid values")
    pooled_row_uids = set(pooled_core14_predictions["row_uid"].astype(str))
    per_waste_row_uids = set(predictions["row_uid"].astype(str))
    if per_waste_row_uids != pooled_row_uids:
        raise RuntimeError("per-waste prediction row_uid set does not match pooled core14 prediction row_uid set")
    metrics_by_fold = pd.read_csv(METRICS_BY_FOLD_PATH)
    expected_metric_pairs = {(scope, waste, int(fold)) for scope in ("pooled_core14", "per_waste_core10") for waste in WASTE_TYPES for fold in OUTER_FOLDS}
    actual_metric_pairs = set(
        zip(
            metrics_by_fold["model_scope"].astype(str),
            metrics_by_fold["WasteFlag"].astype(str),
            metrics_by_fold["outer_fold"].astype(int),
        )
    )
    if actual_metric_pairs != expected_metric_pairs:
        raise RuntimeError("bridge metrics by fold must cover both scopes, four wastes, and five folds")
    artifacts = pd.read_csv(ARTIFACT_MANIFEST_PATH)
    if len(artifacts) != len(WASTE_TYPES) * len(OUTER_FOLDS):
        raise RuntimeError("artifact manifest must contain one per-waste model per outer fold")
    for artifact_path in artifacts["artifact_path"].astype(str):
        require_file(Path(artifact_path))
    diagnostics = pd.read_csv(DIAGNOSTICS_PATH)
    if sorted(diagnostics["WasteFlag"].astype(str).tolist()) != sorted(WASTE_TYPES):
        raise RuntimeError("heterogeneity diagnostics must contain exactly one row per waste type")
    print({"step": "output_assertions_passed", "prediction_rows": int(len(predictions)), "metric_rows": int(len(metrics_by_fold))})


assert_expected_outputs()
